In [1]:
from pathlib import Path

import joblib
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv("/content/support_tickets.csv")

In [3]:
required_columns = {"text", "label"}
missing_columns = required_columns.difference(df.columns)

if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {sorted(missing_columns)}")

print("Missing values before cleaning:")
print(df[["text", "label"]].isna().sum())

df_clean = df.dropna(subset=["text", "label"]).copy()
df_clean["text"] = df_clean["text"].astype(str).str.strip()
df_clean["label"] = df_clean["label"].astype(str).str.strip()
df_clean = df_clean[(df_clean["text"] != "") & (df_clean["label"] != "")]

label_counts_per_text = df_clean.groupby("text")["label"].nunique()
conflicting_texts = label_counts_per_text[label_counts_per_text > 1]

print(f"Texts with conflicting labels: {len(conflicting_texts)}")

if not conflicting_texts.empty:
    display(
        df_clean[df_clean["text"].isin(conflicting_texts.index)]
        .sort_values("text")
        .head(20)
    )
    raise ValueError(
        "Resolve the conflicting labels shown above before training."
    )

Missing values before cleaning:
text     0
label    0
dtype: int64
Texts with conflicting labels: 0


In [4]:
rows_before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset="text", keep="first").copy()

print(f"Rows before deduplication: {rows_before}")
print(f"Rows after deduplication:  {len(df_clean)}")
print(f"Duplicate texts remaining: {df_clean['text'].duplicated().sum()}")

print("\nCategory counts after deduplication:")
print(df_clean["label"].value_counts())

Rows before deduplication: 2000
Rows after deduplication:  826
Duplicate texts remaining: 0

Category counts after deduplication:
label
feature_request    234
bug                227
general_inquiry    155
billing_issue      135
account_problem     75
Name: count, dtype: int64


In [5]:
X = df_clean["text"]
y = df_clean["label"]

print(f"Number of examples: {len(X)}")
print(f"Number of categories: {y.nunique()}")

Number of examples: 826
Number of categories: 5


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Training examples: {len(X_train)}")
print(f"Test examples:     {len(X_test)}")

print("\nTraining category proportions:")
print(y_train.value_counts(normalize=True).sort_index().round(3))

print("\nTest category proportions:")
print(y_test.value_counts(normalize=True).sort_index().round(3))

Training examples: 660
Test examples:     166

Training category proportions:
label
account_problem    0.091
billing_issue      0.164
bug                0.274
feature_request    0.283
general_inquiry    0.188
Name: proportion, dtype: float64

Test category proportions:
label
account_problem    0.090
billing_issue      0.163
bug                0.277
feature_request    0.283
general_inquiry    0.187
Name: proportion, dtype: float64


In [7]:
pipeline = Pipeline(steps=[
    ("tfidf", TfidfVectorizer(max_features=5000, stop_words="english")),
    ("logreg", LogisticRegression(max_iter=1000, random_state=42)),
])

pipeline.fit(X_train, y_train)
print("Training complete.")

Training complete.


In [8]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)

print(f"Accuracy:          {accuracy_score(y_test, y_pred):.3f}")
print(
    f"Precision (macro): "
    f"{precision_score(y_test, y_pred, average='macro', zero_division=0):.3f}"
)
print(
    f"Recall (macro):    "
    f"{recall_score(y_test, y_pred, average='macro', zero_division=0):.3f}"
)
print(f"F1 (macro):        {f1_score(y_test, y_pred, average='macro', zero_division=0):.3f}")
print(f"Log loss:          {log_loss(y_test, y_proba, labels=pipeline.classes_):.3f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))

labels = pipeline.classes_
confusion = pd.DataFrame(
    confusion_matrix(y_test, y_pred, labels=labels),
    index=pd.Index(labels, name="Actual"),
    columns=pd.Index(labels, name="Predicted"),
)

print("Confusion matrix:")
display(confusion)

Accuracy:          0.952
Precision (macro): 0.960
Recall (macro):    0.933
F1 (macro):        0.945
Log loss:          0.324

Classification report:
                 precision    recall  f1-score   support

account_problem       1.00      0.87      0.93        15
  billing_issue       1.00      0.93      0.96        27
            bug       0.94      1.00      0.97        46
feature_request       0.96      1.00      0.98        47
general_inquiry       0.90      0.87      0.89        31

       accuracy                           0.95       166
      macro avg       0.96      0.93      0.94       166
   weighted avg       0.95      0.95      0.95       166

Confusion matrix:


Predicted,account_problem,billing_issue,bug,feature_request,general_inquiry
Actual,,,,,
account_problem,13,0,0,0,2
billing_issue,0,25,1,0,1
bug,0,0,46,0,0
feature_request,0,0,0,47,0
general_inquiry,0,0,2,2,27


In [9]:
new_tickets = [
    "My card was charged twice for the same order.",
    "I cannot sign in after resetting my password.",
    "The mobile app closes whenever I open the settings page.",
    "How can I change the delivery address for my order?",
    "I would like to cancel my subscription before it renews.",
]

new_predictions = pipeline.predict(new_tickets)

new_results = pd.DataFrame({
    "ticket": new_tickets,
    "predicted_label": new_predictions,
})

display(new_results)

,ticket,predicted_label
0,My card was charged twice for the same order.,billing_issue
1,I cannot sign in after resetting my password.,general_inquiry
2,The mobile app closes whenever I open the sett...,bug
3,How can I change the delivery address for my o...,account_problem
4,I would like to cancel my subscription before ...,feature_request


In [10]:
new_probabilities = pipeline.predict_proba(new_tickets)

probability_table = pd.DataFrame(
    new_probabilities,
    columns=pipeline.classes_,
    index=pd.Index(new_tickets, name="ticket"),
)

display(probability_table.round(3))

,account_problem,billing_issue,bug,feature_request,general_inquiry
ticket,,,,,
My card was charged twice for the same order.,0.046,0.660,0.086,0.094,0.114
I cannot sign in after resetting my password.,0.089,0.158,0.222,0.219,0.312
The mobile app closes whenever I open the settings page.,0.049,0.088,0.670,0.101,0.091
How can I change the delivery address for my order?,0.724,0.058,0.066,0.072,0.080
I would like to cancel my subscription before it renews.,0.048,0.077,0.101,0.648,0.126


In [11]:
error_review = pd.DataFrame({
    "ticket": X_test,
    "actual_label": y_test,
    "predicted_label": y_pred,
})

error_review = error_review[
    error_review["actual_label"] != error_review["predicted_label"]
]

print(f"Misclassified test examples: {len(error_review)} of {len(y_test)}")
display(error_review.head(20))

Misclassified test examples: 8 of 166


,ticket,actual_label,predicted_label
64,Cancel subscription failed.,billing_issue,general_inquiry
1718,How do I search?,general_inquiry,bug
551,How do I export?,general_inquiry,feature_request
5,Delete my account please.,account_problem,general_inquiry
183,Where to find export?,general_inquiry,feature_request
898,Where to find login?,general_inquiry,bug
0,Subscription not renewing properly.,billing_issue,bug
9,"Forgot password, reset not working.",account_problem,general_inquiry


In [12]:
model_path = Path("models/ticket_classifier.pkl")
model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(pipeline, model_path)
print(f"Saved trained pipeline to: {model_path}")

Saved trained pipeline to: models/ticket_classifier.pkl
